#Hotel Supplier Entity Resolution Pipeline

This pipeline performs entity resolution across hotel supplier datasets.
The goal is to identify supplier records that represent the same real-world hotel
and create a unified mapping table.

The pipeline performs data ingestion, data quality validation, geographic blocking, candidate pair generation, similarity scoring, and entity resolution to produce a final mapping dataset linking supplier hotel IDs to a master HQ hotel ID.
The solution is designed to minimize unnecessary comparisons and leverage Spark's distributed processing capabilities for efficient execution.

Key Steps:
1. Data ingestion from CSV
2. Text normalization
3. Data quality validation (latitude / longitude)
4. Geographic blocking to reduce comparisons
5. Candidate pair generation
6. Similarity scoring using Levenshtein distance
7. Entity matching using a similarity threshold
8. Export final mapping dataset

Import Required Libraries:

This step imports the PySpark SQL functions required for data transformation, text normalization, geographic bucketing, and similarity scoring.

Using Spark's built-in functions ensures that all transformations are executed in a distributed manner across the cluster, which improves scalability and performance when processing large datasets.

Key functions used:

- **col** → References DataFrame columns in Spark expressions
- **lower** → Standardizes text values for consistent comparison
- **concat** → Combines values to create geographic bucket identifiers
- **floor** → Generates approximate geographic groups
- **lit** → Adds constant values when constructing strings
- **levenshtein** → Computes edit distance between strings for fuzzy matching
- **length** → Normalizes similarity scores

In [0]:
from pyspark.sql.functions import col, lower, concat, floor, lit
from pyspark.sql.functions import levenshtein, length

Load Supplier Dataset:

In this step, the supplier dataset is loaded from a CSV file stored in Databricks FileStore.

I use Spark's CSV reader to ingest the data into a distributed DataFrame so that the dataset can be processed efficiently across the cluster.

Key options used:

- **header = true** → The first row contains column names
- **inferSchema = true** → Spark automatically detects column data types
- **sep = ","** → Specifies that the file is comma-separated

The resulting DataFrame will be used for all downstream transformations in the pipeline.

In [0]:
DATA_PATH = "/FileStore/tables/suppliers_data.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .load(DATA_PATH)

# Inspect schema and sample records during development
# df.printSchema()
# display(df)

Keeping only needed columns helps reduce memory usage and improves performance.

In [0]:
# Select only required columns
df = df.select(
    "inv_id",
    "supplier",
    "name",
    "address",
    "city",
    "latitude",
    "longitude"
)

Normalize Text Fields: Hotel names and addresses may appear in different formats across suppliers. This normalization step ensures that variations in letter casing (e.g., "Hotel Plaza" vs "hotel plaza") do not affect similarity comparisons during the matching process.

In [0]:
# Convert text fields to lowercase for consistent comparison
df_clean = (
    df.withColumn("name_clean", lower(col("name")))
      .withColumn("address_clean", lower(col("address")))
      .withColumn("city_clean", lower(col("city")))
)

Data Quality Validation: Remove Invalid Coordinates. Some records in the dataset contain invalid geographic coordinates where country names or other text values appear in the latitude or longitude fields.

To prevent casting errors and ensure reliable geographic grouping, the pipeline validates coordinates using a regular expression pattern and removes invalid records before converting them to numeric values.

In [0]:
valid_pattern = r'^-?\d+(\.\d+)?$'

df_valid = df_clean.filter(
    col("latitude").rlike(valid_pattern) &
    col("longitude").rlike(valid_pattern)
)

In [0]:
# Count removed records for transparency

bad_rows = df.count() - df_valid.count()
print("Bad coordinate rows removed:", bad_rows)

Bad coordinate rows removed: 48754


In [0]:
# Convert coordinates to numeric format

df_clean = df_valid.withColumn(
    "latitude", col("latitude").cast("double")
).withColumn(
    "longitude", col("longitude").cast("double")
)

Repartition Data: 

In [0]:
# Repartitioning improves parallel processing performance.

df_clean = df_clean.repartition(8)

Create Geographic Buckets: To avoid comparing every hotel with every other hotel in the dataset, a geographic blocking strategy is applied.

Hotels are grouped into geographic buckets derived from their latitude and longitude values. Only hotels located in the same city and geographic bucket are compared.

This significantly reduces the number of candidate comparisons and improves pipeline performance.

In [0]:
# Nearby hotels are grouped together using latitude and longitude buckets.
# This technique reduces unnecessary pair comparisons.

df_clean = df_clean.withColumn(
    "geo_bucket",
    concat(
        floor(col("latitude") * 10),
        lit("_"),
        floor(col("longitude") * 10)
    )
)

Generate Candidate Hotel Pairs: 

A self-join is performed to generate candidate hotel pairs.

Hotels are only compared if they satisfy the following conditions:

- Located in the same city
- Located in the same geographic bucket
- Unique pair comparison to avoid duplicate evaluations

This step generates the set of potential duplicate hotel pairs that will be evaluated using similarity scoring.

In [0]:
# Only compare hotels in the same city and geographic bucket.


pairs = df_clean.alias("a").join(
    df_clean.alias("b"),
    (col("a.city_clean") == col("b.city_clean")) &
    (col("a.geo_bucket") == col("b.geo_bucket")) &
    (col("a.inv_id") < col("b.inv_id"))
)

In [0]:
# Select required fields for similarity comparison

pairs = pairs.select(
    col("a.inv_id").alias("a_inv_id"),
    col("a.supplier").alias("a_supplier"),
    col("a.name_clean").alias("a_name"),
    col("a.address_clean").alias("a_address"),
    col("b.inv_id").alias("b_inv_id"),
    col("b.supplier").alias("b_supplier"),
    col("b.name_clean").alias("b_name"),
    col("b.address_clean").alias("b_address")
)

Select required fields for similarity comparison: 

Fuzzy matching is applied to evaluate how similar two hotel records are.

The pipeline uses the Levenshtein distance between hotel names and addresses to compute similarity scores.

A weighted scoring approach is applied:

- **70% weight** → Hotel name similarity
- **30% weight** → Address similarity

This produces a final similarity score that indicates how likely two records represent the same real-world hotel.

In [0]:
# We measure similarity between hotel names and addresses
# using Levenshtein distance.

pairs = pairs.withColumn(
    "name_score",
    100 - (levenshtein(col("a_name"), col("b_name")) / length(col("a_name")) * 100)
)

pairs = pairs.withColumn(
    "address_score",
    100 - (levenshtein(col("a_address"), col("b_address")) / length(col("a_address")) * 100)
)

# Weighted final similarity score

pairs = pairs.withColumn(
    "final_score",
    col("name_score") * 0.7 + col("address_score") * 0.3
)

Weighted final similarity score:

A similarity threshold of **75%** is used to determine whether two hotel records represent the same real-world entity.

The final similarity score is calculated using a weighted combination of hotel name similarity (70%) and address similarity (30%).

A threshold of 75 provides a balanced trade-off between precision and recall:

- It allows small variations in hotel names or addresses across suppliers.
- It prevents unrelated hotels from being incorrectly matched.

This value was chosen as a practical threshold commonly used in entity resolution tasks where minor textual differences are expected across data sources.

In [0]:
MATCH_THRESHOLD = 75

matches = pairs.filter(col("final_score") > MATCH_THRESHOLD)

Generate Hotel Mapping:

A final mapping table is created that links supplier hotel identifiers to their corresponding HQ hotel identifier.

Hotels that do not have duplicates are also included to ensure complete coverage of the dataset.

The final output contains:

- **hq_id** → Master hotel identifier
- **inv_id** → Supplier hotel identifier
- **inv_name** → Supplier name

In [0]:
# Each matched group is assigned a master HQ hotel ID.

mapping = matches.select(
    col("a_inv_id").alias("hq_id"),
    col("a_inv_id").alias("inv_id"),
    col("a_supplier").alias("inv_name")
).union(
    matches.select(
        col("a_inv_id").alias("hq_id"),
        col("b_inv_id").alias("inv_id"),
        col("b_supplier").alias("inv_name")
    )
)

In [0]:
# Each matched group is assigned a master HQ hotel ID.

single_hotels = df_clean.select(
    col("inv_id").alias("hq_id"),
    col("inv_id").alias("inv_id"),
    col("supplier").alias("inv_name")
)

mapping = mapping.union(single_hotels).dropDuplicates()

Pipeline Metrics: 

Basic metrics are printed to validate pipeline execution:

- Total number of hotels processed
- Number of candidate hotel pairs generated
- Number of matches identified

These metrics help verify that the entity resolution process executed successfully.


In [0]:
print("Total hotels:", df_clean.count())
print("Candidate pairs:", pairs.count())
print("Matches found:", matches.count())

Total hotels: 83707
Candidate pairs: 1150815
Matches found: 12015


Export Final Mapping

In [0]:
#Export single CSV file for submission

mapping.write.mode("overwrite") \
       .option("header", "true") \
       .csv("/FileStore/hotel_mapping_final_output")
print("Pipeline execution completed successfully :-)")

Pipeline execution completed successfully :-)


Display a small sample of the final hotel mapping table to verify that the pipeline produced the expected results.

In [0]:
display(mapping.limit(20))

hq_id,inv_id,inv_name
535982,535982,supplier1
137794,137794,supplier1
174341,174341,supplier1
328647,328647,supplier1
586120,586120,supplier1
180193,180193,supplier1
173661,173661,supplier1
209941,209941,supplier1
458781,458781,supplier1
495637,495637,supplier1


## Conclusion

In this notebook, I built a PySpark pipeline to identify duplicate hotel records across different suppliers and group them under a single hotel identifier.

The process starts by cleaning the data and validating coordinates to ensure only reliable records are used. To avoid comparing every hotel with every other hotel, geographic bucketing is used so that only nearby hotels are compared. After generating candidate pairs, fuzzy matching is applied on hotel names and addresses to measure how similar two records are.

Based on the similarity score, hotels that likely represent the same real-world property are grouped together under one HQ hotel ID. The final output is a mapping dataset that connects supplier hotel IDs to the unified hotel identifier.

This approach helps create a cleaner and more consistent hotel dataset that can be used for further analysis or integration across multiple supplier systems.